 # 04 — Final Functional Classification



 This notebook builds the final, thesis-ready comparison of functional and

 interpretable classification models for sampled Gaia XP spectra.



 Main goals:



 1. Load sampled spectra and class labels.

 2. Reuse the predefined repeated stratified cross-validation splits.

 3. Re-evaluate selected distance-based functional models.

 4. Re-evaluate functional linear models directly on spectra.

 5. Import FPCA/FPLS summary results from the previous notebook when available.

 6. Combine all final results into one common summary table.

 7. Save the interpretability payload used in the later interpretation notebook.



 This notebook does not create figures. It saves CSV tables and one `.npz`

 interpretability payload.

In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")



 ## 1. General configuration



 This section defines the main experiment settings.



 - `RANDOM_STATE` keeps repeated operations reproducible.

 - `SMOKE` can be set to `True` to run only the first repeated CV block.

 - `DISTANCE_K` defines the number of neighbours for weighted kNN.

 - `DISTANCE_MODELS` lists the selected distance-based functional models.

 - `LINEAR_MODELS` lists the final functional linear models.

 - `INNER_CV` and `CS_GRID` define the inner hyperparameter search for linear

   models.

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SMOKE = False

# Number of neighbours used by the selected kNN models.
DISTANCE_K = 5

# Selected distance-based models rerun on the shared repeated CV splits.
DISTANCE_MODELS = [
    {"model_name": "weighted kNN", "distance": "euclidean", "weighted": True},
    {"model_name": "weighted kNN", "distance": "cosine", "weighted": True},
    {"model_name": "weighted kNN", "distance": "seuclidean", "weighted": True},
    {"model_name": "weighted kNN", "distance": "correlation", "weighted": True},
    {"model_name": "centroid", "distance": "euclidean", "weighted": False},
    {"model_name": "centroid", "distance": "cosine", "weighted": False},
    {"model_name": "centroid", "distance": "seuclidean", "weighted": False},
    {"model_name": "centroid", "distance": "correlation", "weighted": False},
]

# Functional linear models fitted directly on sampled spectra.
LINEAR_MODELS = [
    {
        "family": "Functional linear",
        "method": "Functional logistic regression (L2)",
        "kind": "logreg_l2",
    },
    {
        "family": "Functional linear",
        "method": "Functional logistic regression (L1)",
        "kind": "logreg_l1",
    },
    {
        "family": "Functional linear",
        "method": "Functional linear SVM",
        "kind": "linear_svm",
    },
]

# Smaller inner CV and C-grid are used to keep the notebook practical to run.
INNER_CV = 3
CS_GRID = np.logspace(-2, 2, 6)



 ## 2. Paths and input files



 The notebook expects the core input files to be stored in `og_data`.



 Required files:



 - `og_xp.csv`: labels and `source_id`;

 - `xp_sampled_spectra.csv`: sampled spectra;

 - `splits_rskf.json`: predefined repeated stratified CV splits.



 Outputs are saved to `results/04_final_functional_models`.

In [3]:
BASE_DIR = Path.cwd() / "og_data"
OUT_DIR = Path.cwd() / "results" / "04_final_functional_models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FPCA_FPLS_SUMMARY_FILE = OUT_DIR / "fpca_fpls_summary.csv"



 ## 3. Helper functions



 These functions handle file lookup, split ordering, metric calculation,

 result summarisation, and score thresholding.

In [4]:
def find_first_existing(paths: list[Path]) -> Path | None:
    """Return the first existing path from a list of candidate paths.

    Parameters
    ----------
    paths:
        Candidate file paths.

    Returns
    -------
    Path | None
        First existing path, or `None` if none of the paths exists.
    """
    for path in paths:
        if path.exists():
            return path

    return None


def split_sort_key(split_name: str) -> tuple[int, int]:
    """Sort repeated CV split names by repetition and fold number.

    The expected split-name format is similar to `rep0_fold0`.

    Parameters
    ----------
    split_name:
        Name of one repeated cross-validation split.

    Returns
    -------
    tuple[int, int]
        Parsed repetition number and fold number.
    """
    rep = int(split_name.split("_")[0].replace("rep", ""))
    fold = int(split_name.split("_")[1].replace("fold", ""))

    return rep, fold


def mean_std_string(mean: float, std: float) -> str | float:
    """Format a metric as `mean ± std`.

    Parameters
    ----------
    mean:
        Mean value of the metric.
    std:
        Standard deviation of the metric.

    Returns
    -------
    str | float
        Formatted string, or `np.nan` if the mean is missing.
    """
    if pd.isna(mean):
        return np.nan

    std_value = 0.0 if pd.isna(std) else std
    return f"{mean:.4f} ± {std_value:.4f}"


def normalize_scores_train_ref(
    scores_te: np.ndarray,
    scores_tr: np.ndarray,
) -> np.ndarray:
    """Normalize scores using the training-score range.

    The test scores are scaled with the minimum and maximum values observed in
    the training fold. This avoids using test-fold information during
    threshold selection.

    Parameters
    ----------
    scores_te:
        Scores that should be normalised.
    scores_tr:
        Training scores used as the reference range.

    Returns
    -------
    np.ndarray
        Scores clipped to the interval `[0, 1]`.
    """
    lo = float(np.min(scores_tr))
    hi = float(np.max(scores_tr))

    if hi == lo:
        return np.full_like(scores_te, 0.5, dtype=np.float64)

    normalized = (scores_te - lo) / (hi - lo)
    normalized = np.clip(normalized, 0.0, 1.0).astype(np.float64)

    return normalized


def pick_youden_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    grid_size: int = 200,
) -> float:
    """Choose the threshold that maximises Youden's J statistic.

    Youden's J is calculated as sensitivity + specificity - 1. The threshold is
    selected on the training fold and then applied to the test fold.

    Parameters
    ----------
    y_true:
        True binary labels.
    y_prob:
        Normalised class-1 scores.
    grid_size:
        Number of candidate thresholds between 0 and 1.

    Returns
    -------
    float
        Best threshold according to Youden's J.
    """
    thresholds = np.linspace(0, 1, grid_size)
    best_j = -1.0
    best_threshold = 0.5

    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(np.int64)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
        youden_j = sensitivity + specificity - 1.0

        if youden_j > best_j:
            best_j = youden_j
            best_threshold = float(threshold)

    return best_threshold


def fold_metrics(
    y_true_te: np.ndarray,
    y_score_te: np.ndarray,
    y_true_tr: np.ndarray,
    y_score_tr: np.ndarray,
) -> dict[str, float]:
    """Calculate all fold-level classification metrics.

    PR-AUC and ROC-AUC are calculated from continuous scores. Binary metrics
    are calculated after choosing a Youden threshold on the training fold.

    Parameters
    ----------
    y_true_te:
        True labels for the test fold.
    y_score_te:
        Continuous model scores for the test fold.
    y_true_tr:
        True labels for the training fold.
    y_score_tr:
        Continuous model scores for the training fold.

    Returns
    -------
    dict[str, float]
        Fold-level metric dictionary.
    """
    metrics = {
        "pr_auc": average_precision_score(y_true_te, y_score_te),
    }

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true_te, y_score_te))
    except ValueError:
        metrics["roc_auc"] = np.nan

    prob_tr = normalize_scores_train_ref(y_score_tr, y_score_tr)
    prob_te = normalize_scores_train_ref(y_score_te, y_score_tr)

    threshold = pick_youden_threshold(y_true_tr, prob_tr)
    y_pred = (prob_te >= threshold).astype(np.int64)

    metrics["youden_threshold"] = threshold
    metrics["sensitivity"] = recall_score(
        y_true_te,
        y_pred,
        pos_label=1,
        zero_division=0,
    )
    metrics["precision"] = precision_score(
        y_true_te,
        y_pred,
        pos_label=1,
        zero_division=0,
    )
    metrics["specificity"] = recall_score(
        y_true_te,
        y_pred,
        pos_label=0,
        zero_division=0,
    )
    metrics["accuracy"] = accuracy_score(y_true_te, y_pred)
    metrics["f1"] = f1_score(
        y_true_te,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true_te,
        y_pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    metrics["youden_j"] = sensitivity + specificity - 1.0

    return metrics


def summarise_run(
    df_run: pd.DataFrame,
    group_cols: list[str],
) -> pd.DataFrame:
    """Summarise fold-level metrics by selected grouping columns.

    Parameters
    ----------
    df_run:
        Fold-level metric table.
    group_cols:
        Columns used for grouping the fold-level results.

    Returns
    -------
    pd.DataFrame
        Summary table with mean and standard deviation for each metric.
    """
    metric_cols = [
        "pr_auc",
        "roc_auc",
        "sensitivity",
        "precision",
        "specificity",
        "accuracy",
        "f1",
        "youden_j",
        "youden_threshold",
    ]

    agg_dict = {}

    for metric in metric_cols:
        agg_dict[f"{metric}_mean"] = pd.NamedAgg(
            column=metric,
            aggfunc="mean",
        )
        agg_dict[f"{metric}_std"] = pd.NamedAgg(
            column=metric,
            aggfunc="std",
        )

    return df_run.groupby(group_cols).agg(**agg_dict).reset_index()



 ## 4. Locate core input files



 This section checks that all required files are available before running the

 final model comparison.

In [5]:
LABEL_FILE = find_first_existing([BASE_DIR / "og_xp.csv"])
SPLIT_FILE = find_first_existing([BASE_DIR / "splits_rskf.json"])
SAMPLED_FILE = find_first_existing([BASE_DIR / "xp_sampled_spectra.csv"])

if LABEL_FILE is None:
    raise FileNotFoundError("Missing `og_xp.csv`.")

if SPLIT_FILE is None:
    raise FileNotFoundError("Missing `splits_rskf.json`.")

if SAMPLED_FILE is None:
    raise FileNotFoundError("Missing `xp_sampled_spectra.csv`.")

print("LABEL_FILE:", LABEL_FILE)
print("SPLIT_FILE:", SPLIT_FILE)
print("SAMPLED_FILE:", SAMPLED_FILE)
print(
    "FPCA/FPLS summary:",
    FPCA_FPLS_SUMMARY_FILE if FPCA_FPLS_SUMMARY_FILE.exists() else "not found",
)



LABEL_FILE: c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\og_data\og_xp.csv
SPLIT_FILE: c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\og_data\splits_rskf.json
SAMPLED_FILE: c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\og_data\xp_sampled_spectra.csv
FPCA/FPLS summary: not found


 ## 5. Load labels, sampled spectra, and repeated splits



 Labels and sampled spectra are merged by `source_id`. Spectra are then

 converted to a numeric matrix and row-wise L2-normalised.

In [6]:
df_labels = pd.read_csv(LABEL_FILE)

if "source_id" not in df_labels.columns or "y" not in df_labels.columns:
    raise ValueError("`og_xp.csv` must contain `source_id` and `y`.")

df_spec = pd.read_csv(SAMPLED_FILE)

if "source_id" not in df_spec.columns:
    raise ValueError("`xp_sampled_spectra.csv` must contain `source_id`.")

wl_cols = [col for col in df_spec.columns if col.startswith("wl_")]

if len(wl_cols) == 0:
    raise ValueError("No sampled spectrum columns found. Expected `wl_*`.")

wavelengths = np.array(
    [float(col.split("_")[1]) for col in wl_cols],
    dtype=np.float64,
)

df_m = df_labels[["source_id", "y"]].merge(
    df_spec[["source_id"] + wl_cols],
    on="source_id",
    how="inner",
    validate="one_to_one",
)

X_raw = df_m[wl_cols].to_numpy(dtype=np.float64)
y = df_m["y"].to_numpy(dtype=np.int64)

# L2-normalise each spectrum so models focus on shape rather than flux scale.
row_norm = np.linalg.norm(X_raw, axis=1, keepdims=True)
X = np.divide(
    X_raw,
    row_norm,
    out=np.zeros_like(X_raw),
    where=row_norm > 1e-20,
)

with open(SPLIT_FILE, encoding="utf-8") as file:
    splits = json.load(file)

split_names = sorted(splits.keys(), key=split_sort_key)

if SMOKE:
    split_names = [
        split_name
        for split_name in split_names
        if split_name.startswith("rep0_")
    ]

print("Merged rows:", len(df_m))
print("X shape:", X.shape)
print("Binary fraction:", round((y == 1).mean(), 4))
print("Number of splits:", len(split_names))
print("Wavelength range:", wavelengths.min(), "to", wavelengths.max())



Merged rows: 2815
X shape: (2815, 343)
Binary fraction: 0.1982
Number of splits: 50
Wavelength range: 336.0 to 1020.0


 ## 6. Distance-model helper functions



 These functions implement the final selected distance-based functional

 classifiers: weighted kNN and centroid classifiers with selected distance

 metrics.

In [7]:
def train_distance_params(
    X_tr: np.ndarray,
    distance_name: str,
) -> dict[str, object]:
    """Estimate training-only parameters needed by distance metrics.

    Standardized Euclidean distance needs feature-wise variances estimated from
    the training fold. Other selected distances do not require additional
    stored parameters.

    Parameters
    ----------
    X_tr:
        Training feature matrix.
    distance_name:
        Distance metric name.

    Returns
    -------
    dict[str, object]
        Distance configuration and any training-derived parameters.
    """
    params = {"distance": distance_name}

    if distance_name == "seuclidean":
        variances = np.var(X_tr, axis=0, ddof=1)
        variances = np.where(variances <= 1e-12, 1e-12, variances)
        params["V"] = variances

    elif distance_name == "correlation":
        params["mu"] = np.mean(X_tr, axis=0)
        params["sigma"] = np.std(X_tr, axis=0) + 1e-12

    return params


def pairwise_distance_matrix(
    X_a: np.ndarray,
    X_b: np.ndarray,
    params: dict[str, object],
) -> np.ndarray:
    """Calculate a pairwise distance matrix using stored metric parameters.

    Parameters
    ----------
    X_a:
        First feature matrix.
    X_b:
        Second feature matrix.
    params:
        Distance parameter dictionary returned by `train_distance_params`.

    Returns
    -------
    np.ndarray
        Pairwise distance matrix.
    """
    distance_name = params["distance"]

    if distance_name == "euclidean":
        return pairwise_distances(X_a, X_b, metric="euclidean")

    if distance_name == "cosine":
        return pairwise_distances(X_a, X_b, metric="cosine")

    if distance_name == "seuclidean":
        return pairwise_distances(
            X_a,
            X_b,
            metric="seuclidean",
            V=params["V"],
        )

    if distance_name == "correlation":
        return pairwise_distances(X_a, X_b, metric="correlation")

    raise ValueError(f"Unsupported distance: {distance_name}")


def knn_scores_from_distances(
    distances: np.ndarray,
    y_ref: np.ndarray,
    k: int,
    weighted: bool = True,
    exclude_self: bool = False,
) -> np.ndarray:
    """Convert a distance matrix into kNN class-1 scores.

    For training-fold scores, `exclude_self=True` removes the zero-distance
    self-neighbour from the neighbour set.

    Parameters
    ----------
    distances:
        Distance matrix between query objects and reference objects.
    y_ref:
        Labels of the reference objects.
    k:
        Number of neighbours.
    weighted:
        Whether to use inverse-distance weighting.
    exclude_self:
        Whether to exclude diagonal self-distances.

    Returns
    -------
    np.ndarray
        Continuous class-1 scores.
    """
    distances = distances.copy()

    if exclude_self and distances.shape[0] == distances.shape[1]:
        np.fill_diagonal(distances, np.inf)

    nn_idx = np.argsort(distances, axis=1)[:, :k]
    nn_distances = np.take_along_axis(distances, nn_idx, axis=1)
    nn_labels = y_ref[nn_idx]

    if weighted:
        weights = 1.0 / np.maximum(nn_distances, 1e-12)
        scores = (weights * nn_labels).sum(axis=1) / weights.sum(axis=1)
    else:
        scores = nn_labels.mean(axis=1)

    return scores.astype(np.float64)


def centroid_scores_from_distances(
    X_ref: np.ndarray,
    y_ref: np.ndarray,
    X_query: np.ndarray,
    params: dict[str, object],
) -> np.ndarray:
    """Calculate centroid-based class-1 scores.

    Each class is represented by its mean spectrum in the training fold. The
    score is higher when the query object is closer to the binary centroid.

    Parameters
    ----------
    X_ref:
        Training feature matrix used to calculate class centroids.
    y_ref:
        Training labels.
    X_query:
        Objects to score.
    params:
        Distance parameter dictionary.

    Returns
    -------
    np.ndarray
        Continuous class-1 scores.
    """
    centroid_0 = X_ref[y_ref == 0].mean(axis=0, keepdims=True)
    centroid_1 = X_ref[y_ref == 1].mean(axis=0, keepdims=True)

    distance_0 = pairwise_distance_matrix(
        X_query,
        centroid_0,
        params,
    ).ravel()
    distance_1 = pairwise_distance_matrix(
        X_query,
        centroid_1,
        params,
    ).ravel()

    similarity_0 = 1.0 / np.maximum(distance_0, 1e-12)
    similarity_1 = 1.0 / np.maximum(distance_1, 1e-12)

    score = similarity_1 / (similarity_0 + similarity_1)

    return score.astype(np.float64)


def run_distance_model_one_split(
    X: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    model_cfg: dict[str, object],
) -> dict[str, object]:
    """Run one distance-based model on one CV split.

    Parameters
    ----------
    X:
        Full L2-normalised spectra matrix.
    y:
        Full binary label vector.
    train_idx:
        Training indices for the current split.
    test_idx:
        Test indices for the current split.
    model_cfg:
        Configuration of the selected distance-based model.

    Returns
    -------
    dict[str, object]
        One fold-level result row.
    """
    X_tr = X[train_idx]
    X_te = X[test_idx]
    y_tr = y[train_idx]
    y_te = y[test_idx]

    params = train_distance_params(X_tr, model_cfg["distance"])

    if model_cfg["model_name"] == "weighted kNN":
        distance_train = pairwise_distance_matrix(X_tr, X_tr, params)
        distance_test = pairwise_distance_matrix(X_te, X_tr, params)

        score_tr = knn_scores_from_distances(
            distance_train,
            y_tr,
            k=DISTANCE_K,
            weighted=model_cfg["weighted"],
            exclude_self=True,
        )
        score_te = knn_scores_from_distances(
            distance_test,
            y_tr,
            k=DISTANCE_K,
            weighted=model_cfg["weighted"],
            exclude_self=False,
        )

    elif model_cfg["model_name"] == "centroid":
        score_tr = centroid_scores_from_distances(
            X_tr,
            y_tr,
            X_tr,
            params,
        )
        score_te = centroid_scores_from_distances(
            X_tr,
            y_tr,
            X_te,
            params,
        )

    else:
        raise ValueError(f"Unsupported model: {model_cfg['model_name']}")

    metrics = fold_metrics(y_te, score_te, y_tr, score_tr)

    output = {
        "family": "Distance-based functional",
        "model_name": model_cfg["model_name"],
        "distance": model_cfg["distance"],
        "method": f"{model_cfg['model_name']} + {model_cfg['distance']}",
    }
    output.update(metrics)

    return output



 ## 7. Rerun distance-based functional models



 All selected distance-based models are evaluated on the same predefined

 repeated stratified CV splits.

In [8]:
distance_records = []

for split_name in split_names:
    print(f"Current fold: {split_name}")

    train_idx = np.array(splits[split_name]["train"], dtype=int)
    test_idx = np.array(splits[split_name]["test"], dtype=int)

    for model_cfg in DISTANCE_MODELS:
        print(f"  -> {model_cfg['model_name']} + {model_cfg['distance']}")

        record = run_distance_model_one_split(
            X,
            y,
            train_idx,
            test_idx,
            model_cfg,
        )
        record["split"] = split_name
        distance_records.append(record)

df_distance_fold = pd.DataFrame(distance_records)

df_distance_summary = summarise_run(
    df_distance_fold,
    ["family", "model_name", "distance", "method"],
)

print("Distance fold metrics:", df_distance_fold.shape)
print("Distance summary:", df_distance_summary.shape)

display(
    df_distance_summary.sort_values(
        ["f1_mean", "pr_auc_mean"],
        ascending=False,
    ).head(10)
)



Current fold: rep0_fold0
  -> weighted kNN + euclidean
  -> weighted kNN + cosine
  -> weighted kNN + seuclidean
  -> weighted kNN + correlation
  -> centroid + euclidean
  -> centroid + cosine
  -> centroid + seuclidean
  -> centroid + correlation
Current fold: rep0_fold1
  -> weighted kNN + euclidean
  -> weighted kNN + cosine
  -> weighted kNN + seuclidean
  -> weighted kNN + correlation
  -> centroid + euclidean
  -> centroid + cosine
  -> centroid + seuclidean
  -> centroid + correlation
Current fold: rep0_fold2
  -> weighted kNN + euclidean
  -> weighted kNN + cosine
  -> weighted kNN + seuclidean
  -> weighted kNN + correlation
  -> centroid + euclidean
  -> centroid + cosine
  -> centroid + seuclidean
  -> centroid + correlation
Current fold: rep0_fold3
  -> weighted kNN + euclidean
  -> weighted kNN + cosine
  -> weighted kNN + seuclidean
  -> weighted kNN + correlation
  -> centroid + euclidean
  -> centroid + cosine
  -> centroid + seuclidean
  -> centroid + correlation
Curr

,family,model_name,distance,method,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,sensitivity_mean,sensitivity_std,...,specificity_mean,specificity_std,accuracy_mean,accuracy_std,f1_mean,f1_std,youden_j_mean,youden_j_std,youden_threshold_mean,youden_threshold_std
7,Distance-based functional,weighted kNN,seuclidean,weighted kNN + seuclidean,0.808304,0.035406,0.891048,0.020077,0.776499,0.040170,...,0.942927,0.014496,0.909947,0.011811,0.773768,0.027378,0.719426,0.038035,0.282111,0.076271
6,Distance-based functional,weighted kNN,euclidean,weighted kNN + euclidean,0.736810,0.040613,0.854622,0.024432,0.685487,0.041446,...,0.931499,0.016621,0.882735,0.014043,0.698724,0.031923,0.616987,0.041082,0.305829,0.075045
5,Distance-based functional,weighted kNN,cosine,weighted kNN + cosine,0.736709,0.040452,0.854638,0.024412,0.685845,0.042392,...,0.929462,0.018436,0.881172,0.014569,0.696108,0.031373,0.615307,0.040626,0.313367,0.059862
3,Distance-based functional,centroid,seuclidean,centroid + seuclidean,0.573751,0.041169,0.799785,0.024732,0.734780,0.037229,...,0.768679,0.024407,0.761954,0.020622,0.550740,0.028079,0.503458,0.043671,0.337186,0.008689
1,Distance-based functional,centroid,cosine,centroid + cosine,0.539304,0.043801,0.777023,0.025532,0.720800,0.036209,...,0.727871,0.025907,0.726465,0.022095,0.511395,0.027299,0.448671,0.044905,0.310854,0.007598
2,Distance-based functional,centroid,euclidean,centroid + euclidean,0.540099,0.043804,0.777215,0.025559,0.719723,0.036629,...,0.727871,0.026036,0.726252,0.022256,0.510830,0.027600,0.447594,0.045437,0.354070,0.013206
4,Distance-based functional,weighted kNN,correlation,weighted kNN + correlation,0.545041,0.043815,0.728922,0.024417,0.484649,0.060223,...,0.886980,0.045307,0.807211,0.031104,0.499961,0.041552,0.371629,0.050565,0.251859,0.044097
0,Distance-based functional,centroid,correlation,centroid + correlation,0.372718,0.044441,0.677382,0.031218,0.709334,0.046406,...,0.542851,0.041612,0.575844,0.029476,0.398912,0.019379,0.252185,0.042450,0.399196,0.030392


 ## 8. Functional linear model helper functions



 These functions train logistic regression and linear SVM models directly on

 standardised sampled spectra.

In [9]:
def mean_inner_cv_roc_auc_at_best_c(clf: LogisticRegressionCV) -> float:
    """Return mean inner-CV ROC-AUC for the selected logistic-regression C.

    Parameters
    ----------
    clf:
        Fitted `LogisticRegressionCV` model.

    Returns
    -------
    float
        Mean inner cross-validation ROC-AUC at the selected value of `C`.
    """
    scores = clf.scores_

    if scores is None:
        return np.nan

    if isinstance(scores, dict):
        score_array = np.asarray(next(iter(scores.values())))
    elif hasattr(scores, "ndim") and scores.ndim == 3:
        score_array = scores[0]
    else:
        score_array = np.asarray(scores)

    c_grid = np.asarray(clf.Cs_)
    best_c = float(clf.C_[0])
    best_idx = int(np.argmin(np.abs(c_grid - best_c)))

    return float(np.mean(score_array[:, best_idx]))


def run_linear_model_one_split(
    X: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    kind: str,
) -> tuple[dict[str, float], np.ndarray]:
    """Fit one functional linear model on one CV split.

    The model is fitted on standardised spectra. Standardisation is estimated
    on the training fold and then applied to the test fold.

    Parameters
    ----------
    X:
        Full L2-normalised spectra matrix.
    y:
        Full binary label vector.
    train_idx:
        Training indices for the current split.
    test_idx:
        Test indices for the current split.
    kind:
        Model type: `logreg_l2`, `logreg_l1`, or `linear_svm`.

    Returns
    -------
    tuple[dict[str, float], np.ndarray]
        Fold-level metrics and fitted model coefficients.
    """
    X_tr = X[train_idx]
    X_te = X[test_idx]
    y_tr = y[train_idx]
    y_te = y[test_idx]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)

    if kind == "logreg_l2":
        model = LogisticRegressionCV(
            Cs=CS_GRID,
            cv=StratifiedKFold(
                n_splits=INNER_CV,
                shuffle=True,
                random_state=RANDOM_STATE,
            ),
            penalty="l2",
            solver="lbfgs",
            class_weight="balanced",
            scoring="roc_auc",
            max_iter=2000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        model.fit(X_tr_scaled, y_tr)

        score_tr = model.predict_proba(X_tr_scaled)[:, 1]
        score_te = model.predict_proba(X_te_scaled)[:, 1]
        coef = model.coef_[0].copy()

        extra = {
            "best_C": float(model.C_[0]),
            "best_cv_roc_auc": mean_inner_cv_roc_auc_at_best_c(model),
            "n_nonzero_coefs": int(X.shape[1]),
        }

    elif kind == "logreg_l1":
        model = LogisticRegressionCV(
            Cs=CS_GRID,
            cv=StratifiedKFold(
                n_splits=INNER_CV,
                shuffle=True,
                random_state=RANDOM_STATE,
            ),
            penalty="l1",
            solver="liblinear",
            class_weight="balanced",
            scoring="roc_auc",
            max_iter=4000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        model.fit(X_tr_scaled, y_tr)

        score_tr = model.predict_proba(X_tr_scaled)[:, 1]
        score_te = model.predict_proba(X_te_scaled)[:, 1]
        coef = model.coef_[0].copy()

        extra = {
            "best_C": float(model.C_[0]),
            "best_cv_roc_auc": mean_inner_cv_roc_auc_at_best_c(model),
            "n_nonzero_coefs": int(np.sum(np.abs(coef) > 1e-6)),
        }

    elif kind == "linear_svm":
        inner_cv = StratifiedKFold(
            n_splits=INNER_CV,
            shuffle=True,
            random_state=RANDOM_STATE,
        )
        c_scores = []

        # Manual inner-CV loop because LinearSVC does not have a direct
        # LogisticRegressionCV-like wrapper.
        for c_value in CS_GRID:
            fold_scores = []

            for inner_tr, inner_va in inner_cv.split(X_tr_scaled, y_tr):
                clf = LinearSVC(
                    C=c_value,
                    class_weight="balanced",
                    max_iter=10000,
                    random_state=RANDOM_STATE,
                )
                clf.fit(X_tr_scaled[inner_tr], y_tr[inner_tr])

                validation_score = clf.decision_function(
                    X_tr_scaled[inner_va],
                )

                try:
                    auc = roc_auc_score(y_tr[inner_va], validation_score)
                except ValueError:
                    auc = np.nan

                fold_scores.append(auc)

            c_scores.append(np.nanmean(fold_scores))

        best_idx = int(np.nanargmax(c_scores))
        best_c = float(CS_GRID[best_idx])

        model = LinearSVC(
            C=best_c,
            class_weight="balanced",
            max_iter=10000,
            random_state=RANDOM_STATE,
        )
        model.fit(X_tr_scaled, y_tr)

        score_tr = model.decision_function(X_tr_scaled)
        score_te = model.decision_function(X_te_scaled)
        coef = model.coef_[0].copy()

        extra = {
            "best_C": best_c,
            "best_cv_roc_auc": float(c_scores[best_idx]),
            "n_nonzero_coefs": int(np.sum(np.abs(coef) > 1e-12)),
        }

    else:
        raise ValueError(f"Unsupported kind: {kind}")

    metrics = fold_metrics(y_te, score_te, y_tr, score_tr)
    metrics.update(extra)

    return metrics, coef



 ## 9. Rerun functional logistic regression and linear SVM



 Functional linear models are trained directly on the L2-normalised sampled

 spectra. Coefficients are stored for later interpretability use.

In [10]:
linear_records = []
coef_store = {model_cfg["method"]: [] for model_cfg in LINEAR_MODELS}

for split_name in split_names:
    print(f"Current fold: {split_name}")

    train_idx = np.array(splits[split_name]["train"], dtype=int)
    test_idx = np.array(splits[split_name]["test"], dtype=int)

    for model_cfg in LINEAR_MODELS:
        print(f"  -> {model_cfg['method']}")

        metrics, coef = run_linear_model_one_split(
            X,
            y,
            train_idx,
            test_idx,
            model_cfg["kind"],
        )

        row = {
            "split": split_name,
            "family": model_cfg["family"],
            "method": model_cfg["method"],
        }
        row.update(metrics)

        linear_records.append(row)
        coef_store[model_cfg["method"]].append(coef)

df_linear_fold = pd.DataFrame(linear_records)

df_linear_summary = summarise_run(
    df_linear_fold,
    ["family", "method"],
)

print("\nLinear model processing complete.")
print("Linear fold metrics:", df_linear_fold.shape)
print("Linear summary:", df_linear_summary.shape)

display(
    df_linear_summary.sort_values(
        ["f1_mean", "pr_auc_mean"],
        ascending=False,
    )
)



Current fold: rep0_fold0
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep0_fold1
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep0_fold2
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep0_fold3
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep0_fold4
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep1_fold0
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep1_fold1
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep1_fold2
  -> Functional logistic regression (

,family,method,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,sensitivity_mean,sensitivity_std,precision_mean,precision_std,specificity_mean,specificity_std,accuracy_mean,accuracy_std,f1_mean,f1_std,youden_j_mean,youden_j_std,youden_threshold_mean,youden_threshold_std
2,Functional linear,Functional logistic regression (L2),0.865021,0.028002,0.915473,0.019766,0.829204,0.032859,0.792494,0.038873,0.945767,0.012882,0.922664,0.012191,0.809755,0.027759,0.774971,0.035251,0.416583,0.031867
0,Functional linear,Functional linear SVM,0.865660,0.029134,0.915126,0.019682,0.824735,0.031628,0.791597,0.045921,0.945548,0.015165,0.921599,0.013791,0.807011,0.030553,0.770283,0.035378,0.163015,0.023552
1,Functional linear,Functional logistic regression (L1),0.854786,0.030313,0.911486,0.020487,0.827954,0.032369,0.770905,0.050652,0.938058,0.017643,0.916234,0.014862,0.797296,0.031403,0.766012,0.035086,0.459296,0.056489


 ## 10. Import FPCA/FPLS summary



 If the summary from the FPCA/FPLS notebook is available, it is imported and

 combined with the rerun distance-based and functional linear results.

In [11]:
df_fpca_fpls = None

if FPCA_FPLS_SUMMARY_FILE.exists():
    df_fpca_fpls = pd.read_csv(FPCA_FPLS_SUMMARY_FILE).copy()
    print("Loaded FPCA/FPLS summary:", df_fpca_fpls.shape)
else:
    print("FPCA/FPLS summary not found. Final table will include only rerun models.")



FPCA/FPLS summary not found. Final table will include only rerun models.


 ## 11. Convert all result sources into one final schema



 Distance-based, functional linear, and imported FPCA/FPLS results are

 converted to the same column structure before final ranking.

In [12]:
final_rows = []

for _, row in df_distance_summary.iterrows():
    final_rows.append(
        {
            "source": "rerun",
            "family": row["family"],
            "method": row["method"],
            "representation": "L2-normalised sampled spectra",
            "tuning": (
                f"k={DISTANCE_K}"
                if "kNN" in row["model_name"]
                else "class centroids"
            ),
            "pr_auc_mean": row["pr_auc_mean"],
            "pr_auc_std": row["pr_auc_std"],
            "roc_auc_mean": row["roc_auc_mean"],
            "roc_auc_std": row["roc_auc_std"],
            "sensitivity_mean": row["sensitivity_mean"],
            "sensitivity_std": row["sensitivity_std"],
            "precision_mean": row["precision_mean"],
            "precision_std": row["precision_std"],
            "specificity_mean": row["specificity_mean"],
            "specificity_std": row["specificity_std"],
            "accuracy_mean": row["accuracy_mean"],
            "accuracy_std": row["accuracy_std"],
            "f1_mean": row["f1_mean"],
            "f1_std": row["f1_std"],
            "youden_j_mean": row["youden_j_mean"],
            "youden_j_std": row["youden_j_std"],
        }
    )

for _, row in df_linear_summary.iterrows():
    subset = df_linear_fold[df_linear_fold["method"] == row["method"]]
    best_c_mean = subset["best_C"].mean()
    tuning = f"inner-CV C | mean best C={best_c_mean:.4g}"

    final_rows.append(
        {
            "source": "rerun",
            "family": row["family"],
            "method": row["method"],
            "representation": "L2-normalised sampled spectra",
            "tuning": tuning,
            "pr_auc_mean": row["pr_auc_mean"],
            "pr_auc_std": row["pr_auc_std"],
            "roc_auc_mean": row["roc_auc_mean"],
            "roc_auc_std": row["roc_auc_std"],
            "sensitivity_mean": row["sensitivity_mean"],
            "sensitivity_std": row["sensitivity_std"],
            "precision_mean": row["precision_mean"],
            "precision_std": row["precision_std"],
            "specificity_mean": row["specificity_mean"],
            "specificity_std": row["specificity_std"],
            "accuracy_mean": row["accuracy_mean"],
            "accuracy_std": row["accuracy_std"],
            "f1_mean": row["f1_mean"],
            "f1_std": row["f1_std"],
            "youden_j_mean": row["youden_j_mean"],
            "youden_j_std": row["youden_j_std"],
        }
    )

if df_fpca_fpls is not None:
    df_use = df_fpca_fpls.copy()

    if "method_family" in df_use.columns and "classifier" in df_use.columns:
        df_use["family"] = "FPCA / FPLS"
        df_use["method"] = (
            df_use["method_family"].astype(str)
            + " + "
            + df_use["classifier"].astype(str)
        )
        df_use["representation"] = df_use["method_family"].astype(str)
        df_use["tuning"] = "J=" + df_use["J"].astype(str)

        for _, row in df_use.iterrows():
            final_rows.append(
                {
                    "source": "imported",
                    "family": row["family"],
                    "method": row["method"],
                    "representation": row["representation"],
                    "tuning": row["tuning"],
                    "pr_auc_mean": row.get("pr_auc_mean", np.nan),
                    "pr_auc_std": row.get("pr_auc_std", np.nan),
                    "roc_auc_mean": row.get("roc_auc_mean", np.nan),
                    "roc_auc_std": row.get("roc_auc_std", np.nan),
                    "sensitivity_mean": row.get("sensitivity_mean", np.nan),
                    "sensitivity_std": row.get("sensitivity_std", np.nan),
                    "precision_mean": row.get("precision_mean", np.nan),
                    "precision_std": row.get("precision_std", np.nan),
                    "specificity_mean": row.get("specificity_mean", np.nan),
                    "specificity_std": row.get("specificity_std", np.nan),
                    "accuracy_mean": row.get("accuracy_mean", np.nan),
                    "accuracy_std": row.get("accuracy_std", np.nan),
                    "f1_mean": row.get("f1_mean", np.nan),
                    "f1_std": row.get("f1_std", np.nan),
                    "youden_j_mean": row.get("youden_j_mean", np.nan),
                    "youden_j_std": row.get("youden_j_std", np.nan),
                }
            )

final_df = pd.DataFrame(final_rows)

print("Final combined rows:", final_df.shape)



Final combined rows: (11, 21)


 ## 12. Final ranking table



 Models are ranked by F1 first, then PR-AUC and ROC-AUC.

In [13]:
rank_by_f1 = final_df.sort_values(
    ["f1_mean", "pr_auc_mean", "roc_auc_mean"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)

print("\n=== TOP MODELS BY F1 ===")
display(
    rank_by_f1[
        [
            "source",
            "family",
            "method",
            "representation",
            "tuning",
            "f1_mean",
            "pr_auc_mean",
            "roc_auc_mean",
        ]
    ].head(20)
)




=== TOP MODELS BY F1 ===


,source,family,method,representation,tuning,f1_mean,pr_auc_mean,roc_auc_mean
0,rerun,Functional linear,Functional logistic regression (L2),L2-normalised sampled spectra,inner-CV C | mean best C=0.01743,0.809755,0.865021,0.915473
1,rerun,Functional linear,Functional linear SVM,L2-normalised sampled spectra,inner-CV C | mean best C=0.01776,0.807011,0.865660,0.915126
2,rerun,Functional linear,Functional logistic regression (L1),L2-normalised sampled spectra,inner-CV C | mean best C=0.869,0.797296,0.854786,0.911486
3,rerun,Distance-based functional,weighted kNN + seuclidean,L2-normalised sampled spectra,k=5,0.773768,0.808304,0.891048
4,rerun,Distance-based functional,weighted kNN + euclidean,L2-normalised sampled spectra,k=5,0.698724,0.736810,0.854622
5,rerun,Distance-based functional,weighted kNN + cosine,L2-normalised sampled spectra,k=5,0.696108,0.736709,0.854638
6,rerun,Distance-based functional,centroid + seuclidean,L2-normalised sampled spectra,class centroids,0.550740,0.573751,0.799785
7,rerun,Distance-based functional,centroid + cosine,L2-normalised sampled spectra,class centroids,0.511395,0.539304,0.777023
8,rerun,Distance-based functional,centroid + euclidean,L2-normalised sampled spectra,class centroids,0.510830,0.540099,0.777215
9,rerun,Distance-based functional,weighted kNN + correlation,L2-normalised sampled spectra,k=5,0.499961,0.545041,0.728922


 ## 13. Best method within each family



 This table keeps only the best-performing method within each model family.

In [14]:
best_by_family = (
    final_df.sort_values(
        ["f1_mean", "pr_auc_mean"],
        ascending=False,
        na_position="last",
    )
    .groupby("family", as_index=False)
    .first()
)

best_by_family_pretty = pd.DataFrame(
    {
        "Family": best_by_family["family"],
        "Method": best_by_family["method"],
        "Representation": best_by_family["representation"],
        "Tuning": best_by_family["tuning"],
        "PR-AUC mean ± std": [
            mean_std_string(mean, std)
            for mean, std in zip(
                best_by_family["pr_auc_mean"],
                best_by_family["pr_auc_std"],
            )
        ],
        "ROC-AUC mean ± std": [
            mean_std_string(mean, std)
            for mean, std in zip(
                best_by_family["roc_auc_mean"],
                best_by_family["roc_auc_std"],
            )
        ],
        "F1 mean ± std": [
            mean_std_string(mean, std)
            for mean, std in zip(
                best_by_family["f1_mean"],
                best_by_family["f1_std"],
            )
        ],
    }
)

print("\n=== BEST BY FAMILY ===")
display(best_by_family_pretty)




=== BEST BY FAMILY ===


,Family,Method,Representation,Tuning,PR-AUC mean ± std,ROC-AUC mean ± std,F1 mean ± std
0,Distance-based functional,weighted kNN + seuclidean,L2-normalised sampled spectra,k=5,0.8083 ± 0.0354,0.8910 ± 0.0201,0.7738 ± 0.0274
1,Functional linear,Functional logistic regression (L2),L2-normalised sampled spectra,inner-CV C | mean best C=0.01743,0.8650 ± 0.0280,0.9155 ± 0.0198,0.8098 ± 0.0278


 ## 14. Build interpretability payload



 The payload contains the core arrays needed by the later interpretability

 notebook:



 - wavelength grid;

 - class labels;

 - class mean spectra and their difference;

 - 2D FPCA/FPLS scores;

 - one selected query spectrum and its nearest neighbour;

 - fitted logistic regression and linear SVM coefficient vectors.

In [15]:
mean_0 = X[y == 0].mean(axis=0)
mean_1 = X[y == 1].mean(axis=0)
diff_10 = mean_1 - mean_0

# Two-dimensional FPCA scores for later visualisation.
pca = PCA(n_components=2, random_state=RANDOM_STATE)
scores_fpca = pca.fit_transform(X)

# Two-dimensional FPLS scores for later visualisation.
pls = PLSRegression(n_components=2)
scores_fpls, _ = pls.fit_transform(X, y.astype(float).reshape(-1, 1))

# Choose the first binary object from the first test fold as a query example.
first_split = split_names[0]
train_idx = np.array(splits[first_split]["train"], dtype=int)
test_idx = np.array(splits[first_split]["test"], dtype=int)

X_tr = X[train_idx]
X_te = X[test_idx]
y_tr = y[train_idx]
y_te = y[test_idx]

test_positive_positions = np.where(y_te == 1)[0]
example_te_local = (
    int(test_positive_positions[0])
    if len(test_positive_positions)
    else 0
)

x_query = X_te[example_te_local : example_te_local + 1]

# Find the closest training spectrum by Euclidean distance.
distances = pairwise_distances(x_query, X_tr, metric="euclidean").ravel()
nearest_idx = np.argsort(distances)[:DISTANCE_K]
closest_idx = int(nearest_idx[0])
closest_curve = X_tr[closest_idx]
abs_diff = np.abs(x_query.ravel() - closest_curve)

# Refit the main linear models on all data using median best C from CV.
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X)

logreg_l2_best_c = float(
    df_linear_fold.loc[
        df_linear_fold["method"] == "Functional logistic regression (L2)",
        "best_C",
    ].median()
)

svm_best_c = float(
    df_linear_fold.loc[
        df_linear_fold["method"] == "Functional linear SVM",
        "best_C",
    ].median()
)

logreg_full = LogisticRegressionCV(
    Cs=[logreg_l2_best_c],
    cv=3,
    penalty="l2",
    solver="lbfgs",
    class_weight="balanced",
    scoring="roc_auc",
    max_iter=4000,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
logreg_full.fit(X_full_scaled, y)
beta_logreg = logreg_full.coef_[0].copy()

svm_full = LinearSVC(
    C=svm_best_c,
    class_weight="balanced",
    max_iter=10000,
    random_state=RANDOM_STATE,
)
svm_full.fit(X_full_scaled, y)
beta_svm = svm_full.coef_[0].copy()

interpretability_payload_path = OUT_DIR / "interpretability_payload.npz"

np.savez(
    interpretability_payload_path,
    wavelengths=wavelengths,
    y=y,
    mean_0=mean_0,
    mean_1=mean_1,
    diff_10=diff_10,
    scores_fpca=scores_fpca,
    scores_fpls=scores_fpls,
    x_query=x_query.ravel(),
    closest_curve=closest_curve,
    abs_diff=abs_diff,
    beta_logreg=beta_logreg,
    beta_svm=beta_svm,
)

print("Interpretability payload saved:", interpretability_payload_path)



Interpretability payload saved: c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\results\04_final_functional_models\interpretability_payload.npz


 ## 15. Save main outputs



 This notebook does not have a separate figure-saving block because it does

 not create figures. The section below saves all CSV outputs and the

 interpretability payload.

In [16]:
distance_fold_path = OUT_DIR / "final_distance_fold_metrics.csv"
linear_fold_path = OUT_DIR / "final_linear_fold_metrics.csv"
final_summary_path = OUT_DIR / "final_model_summary.csv"
rank_f1_path = OUT_DIR / "final_rank_by_f1.csv"
best_family_path = OUT_DIR / "final_best_by_family.csv"

df_distance_fold.to_csv(distance_fold_path, index=False)
df_linear_fold.to_csv(linear_fold_path, index=False)
final_df.to_csv(final_summary_path, index=False)
rank_by_f1.to_csv(rank_f1_path, index=False)
best_by_family_pretty.to_csv(best_family_path, index=False)

print("Saved:")
print("-", distance_fold_path)
print("-", linear_fold_path)
print("-", final_summary_path)
print("-", rank_f1_path)
print("-", best_family_path)
print("-", interpretability_payload_path)



Saved:
- c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\results\04_final_functional_models\final_distance_fold_metrics.csv
- c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\results\04_final_functional_models\final_linear_fold_metrics.csv
- c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\results\04_final_functional_models\final_model_summary.csv
- c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\results\04_final_functional_models\final_rank_by_f1.csv
- c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\results\04_final_functional_models\final_best_by_family.csv
- c:\Users\Lenovo\Documents\VDA\Bakalauras\Kodas\results\04_final_functional_models\interpretability_payload.npz


 ## 16. Short final view



 This final display repeats the most important ranking outputs for quick

 inspection after the notebook finishes running.

In [17]:
print("\n=== FINAL TOP 15 BY F1 ===")
display(
    rank_by_f1[
        [
            "family",
            "method",
            "representation",
            "tuning",
            "f1_mean",
            "pr_auc_mean",
            "roc_auc_mean",
        ]
    ].head(15)
)

print("\n=== BEST BY FAMILY ===")
display(best_by_family_pretty)


=== FINAL TOP 15 BY F1 ===


,family,method,representation,tuning,f1_mean,pr_auc_mean,roc_auc_mean
0,Functional linear,Functional logistic regression (L2),L2-normalised sampled spectra,inner-CV C | mean best C=0.01743,0.809755,0.865021,0.915473
1,Functional linear,Functional linear SVM,L2-normalised sampled spectra,inner-CV C | mean best C=0.01776,0.807011,0.865660,0.915126
2,Functional linear,Functional logistic regression (L1),L2-normalised sampled spectra,inner-CV C | mean best C=0.869,0.797296,0.854786,0.911486
3,Distance-based functional,weighted kNN + seuclidean,L2-normalised sampled spectra,k=5,0.773768,0.808304,0.891048
4,Distance-based functional,weighted kNN + euclidean,L2-normalised sampled spectra,k=5,0.698724,0.736810,0.854622
5,Distance-based functional,weighted kNN + cosine,L2-normalised sampled spectra,k=5,0.696108,0.736709,0.854638
6,Distance-based functional,centroid + seuclidean,L2-normalised sampled spectra,class centroids,0.550740,0.573751,0.799785
7,Distance-based functional,centroid + cosine,L2-normalised sampled spectra,class centroids,0.511395,0.539304,0.777023
8,Distance-based functional,centroid + euclidean,L2-normalised sampled spectra,class centroids,0.510830,0.540099,0.777215
9,Distance-based functional,weighted kNN + correlation,L2-normalised sampled spectra,k=5,0.499961,0.545041,0.728922



=== BEST BY FAMILY ===


,Family,Method,Representation,Tuning,PR-AUC mean ± std,ROC-AUC mean ± std,F1 mean ± std
0,Distance-based functional,weighted kNN + seuclidean,L2-normalised sampled spectra,k=5,0.8083 ± 0.0354,0.8910 ± 0.0201,0.7738 ± 0.0274
1,Functional linear,Functional logistic regression (L2),L2-normalised sampled spectra,inner-CV C | mean best C=0.01743,0.8650 ± 0.0280,0.9155 ± 0.0198,0.8098 ± 0.0278
